In [1]:
import pandas as pd
import numpy as np
from scipy.linalg import solve


In [2]:
b = pd.read_csv("nl10_buses.csv")
l = pd.read_csv("nl10_lines.csv")
print(b)

    name  v_nom  type         x          y carrier  unit  location  \
0  NL0 0  380.0   NaN  4.218807  51.423433      AC   NaN       NaN   
1  NL0 1  380.0   NaN  6.505559  52.115536      AC   NaN       NaN   
2  NL0 2  380.0   NaN  4.850579  52.354574      AC   NaN       NaN   
3  NL0 3  380.0   NaN  6.754784  53.217513      AC   NaN       NaN   
4  NL0 4  380.0   NaN  5.532565  51.446761      AC   NaN       NaN   
5  NL0 5  380.0   NaN  4.188004  52.048780      AC   NaN       NaN   
6  NL0 6  380.0   NaN  4.737305  51.809433      AC   NaN       NaN   
7  NL0 7  380.0   NaN  5.918315  51.148408      AC   NaN       NaN   
8  NL0 8  380.0   NaN  5.788487  51.782672      AC   NaN       NaN   
9  NL0 9  380.0   NaN  5.885458  52.849781      AC   NaN       NaN   

   v_mag_pu_set  v_mag_pu_min  v_mag_pu_max control  generator  sub_network  \
0           1.0           0.0           inf   Slack        NaN            0   
1           1.0           0.0           inf      PQ        NaN         

In [4]:
# clean the data, keeping only columns we need and get rid of empty rows
buses = b[["name"]].copy()
# drop H2 and battery buses, not relevant for our analysis
buses = buses[~buses["name"].str.contains("H2|battery", case=False, na=False)]


# data needed from lines: name, bus0, bus1, reactance x and capacity s_nom
lines = l[["name", "bus0", "bus1", "x", "r", "s_nom", "s_max_pu"]].copy()
# for line capacity: "s_nom" multplied with "s_max_pu" to recover the usable capacity
lines["s_nom"] = lines["s_nom"] * lines["s_max_pu"]
lines = lines.drop(columns=["s_max_pu"])
# drop lines if value is missing or 0 for any of the columns
lines = lines.dropna(subset=["name", "bus0", "bus1", "x", "r", "s_nom"])
lines = lines[(lines["x"] != 0) & (lines["r"] != 0) & (lines["s_nom"] != 0)]
print(lines)

    name   bus0   bus1          x         r        s_nom
0      0  NL0 0  NL0 6   6.874549  0.838360  2377.343656
1      1  NL0 1  NL0 8   7.567404  0.922854  2377.343656
2     10  NL0 5  NL0 6   2.836244  0.345884  4754.687313
3     11  NL0 7  NL0 8   8.745055  1.066470  2377.343656
4      2  NL0 1  NL0 9  11.292550  1.377140  2377.343656
5      3  NL0 2  NL0 5   6.952542  0.847871  2377.343656
6      4  NL0 2  NL0 6   7.516409  0.916634  2377.343656
7      5  NL0 2  NL0 9  10.943783  1.334608  2377.343656
9      7  NL0 3  NL0 9   3.318029  0.533019  3753.700510
10     8  NL0 4  NL0 6   5.585333  0.681138  3566.015485
11     9  NL0 4  NL0 7   5.247236  0.639907  2377.343656


In [5]:
# reactance-resistance ratio of the lines to check suitability for DC power flow approximation (neglectable ratio > 4)
avg_xr_ratio = (lines["x"] / lines["r"]).mean()
min_xr_ratio = (lines["x"] / lines["r"]).min()
max_xr_ratio = (lines["x"] / lines["r"]).max()
print(f"Average x/r ratio: {avg_xr_ratio:.2f}")
print(f"Minimum x/r ratio: {min_xr_ratio:.2f}")     
print(f"Maximum x/r ratio: {max_xr_ratio:.2f}")

Average x/r ratio: 8.02
Minimum x/r ratio: 6.22
Maximum x/r ratio: 8.20


In [6]:
# indices for buses and lines
buses["bus_idx"] = np.arange(len(buses))
lines["line_idx"] = np.arange(len(lines))
# create a mapping from bus name to bus index
bus_name_to_idx = dict(zip(buses["name"], buses["bus_idx"]))

# map line endpoints to bus indices
lines["bus0_idx"] = lines["bus0"].map(bus_name_to_idx)
lines["bus1_idx"] = lines["bus1"].map(bus_name_to_idx)
print(lines[["name", "bus0", "bus1", "bus0_idx", "bus1_idx"]])

    name   bus0   bus1  bus0_idx  bus1_idx
0      0  NL0 0  NL0 6         0         6
1      1  NL0 1  NL0 8         1         8
2     10  NL0 5  NL0 6         5         6
3     11  NL0 7  NL0 8         7         8
4      2  NL0 1  NL0 9         1         9
5      3  NL0 2  NL0 5         2         5
6      4  NL0 2  NL0 6         2         6
7      5  NL0 2  NL0 9         2         9
9      7  NL0 3  NL0 9         3         9
10     8  NL0 4  NL0 6         4         6
11     9  NL0 4  NL0 7         4         7


In [7]:
# incidence matrix A, where A[l, bus0] = +1 (from) and A[l, bus1] = -1 (to)
num_buses = len(buses)
num_lines = len(lines)
A = np.zeros((num_lines, num_buses), dtype=int)

for _, row in lines.iterrows():
    bus0 = row["bus0_idx"]
    bus1 = row["bus1_idx"]
    l = row["line_idx"]
    A[l, bus0] = 1
    A[l, bus1] = -1
print("Incidence matrix shape:", A.shape)
print(buses[["bus_idx", "name"]].head(10))
display(pd.DataFrame(A[:24, :20]))
row_sums = A.sum(axis=1)
print("\nUnique row sums of A (should be 0):", np.unique(row_sums))

Incidence matrix shape: (11, 10)
   bus_idx   name
0        0  NL0 0
1        1  NL0 1
2        2  NL0 2
3        3  NL0 3
4        4  NL0 4
5        5  NL0 5
6        6  NL0 6
7        7  NL0 7
8        8  NL0 8
9        9  NL0 9


,0,1,2,3,4,5,6,7,8,9
0,1,0,0,0,0,0,-1,0,0,0
1,0,1,0,0,0,0,0,0,-1,0
2,0,0,0,0,0,1,-1,0,0,0
3,0,0,0,0,0,0,0,1,-1,0
4,0,1,0,0,0,0,0,0,0,-1
5,0,0,1,0,0,-1,0,0,0,0
6,0,0,1,0,0,0,-1,0,0,0
7,0,0,1,0,0,0,0,0,0,-1
8,0,0,0,1,0,0,0,0,0,-1
9,0,0,0,0,1,0,-1,0,0,0



Unique row sums of A (should be 0): [0]


In [8]:
# save the ordering tables for later export
buses[["bus_idx", "name"]].to_csv("nl10_bus_order.csv", index=False)
lines[["line_idx", "name", "bus0", "bus1", "x", "s_nom"]].to_csv("nl10_line_order.csv", index=False)

In [9]:
# define reference bus
ref_bus_name = buses.loc[0, "name"]  # choose the first bus as reference
ref_bus_idx = int(buses.loc[buses["name"] == ref_bus_name, "bus_idx"].iloc[0])  # explanation: get the index of the reference bus
print(f"Reference bus: {ref_bus_name} index({ref_bus_idx})")

Reference bus: NL0 0 index(0)


In [10]:
# line susceptance vector b = 1/x
x = lines["x"].to_numpy(dtype=float)
b = 1 / x

# build diagonal matrix B = diag(b)
B = np.diag(b)

# remove the reference bus column from A to get A_red
A_red = np.delete(A, ref_bus_idx, axis=1)

# compute reduced bus susceptance matrix
B_red = A_red.T @ B @ A_red

print("Reduced bus susceptance matrix B_red shape:", B_red.shape)
print("A_red shape:", A_red.shape)

Reduced bus susceptance matrix B_red shape: (9, 9)
A_red shape: (11, 9)


In [11]:
# solve (instead of invert)
PTDF_red = np.linalg.solve(B_red, (B @ A_red).T).T
print("PTDF_red shape:", PTDF_red.shape)

PTDF_red shape: (11, 9)


In [12]:
# insert zero column back at the slack position
PTDF = np.insert(PTDF_red, ref_bus_idx, 0.0, axis=1)
print("Final PTDF shape:", PTDF.shape)


ptdf_df = pd.DataFrame(
    PTDF,
    index=lines["name"],
    columns=buses["name"]
)

# change column names to Bus, row names to Line for easier reference
ptdf_df.columns.name = "Bus"
ptdf_df.index.name = "Line"

display(ptdf_df.iloc[:20, :20])

ptdf_df.to_csv("nl10_ptdf_matrix.csv")

Final PTDF shape: (11, 10)


Bus,NL0 0,NL0 1,NL0 2,NL0 3,NL0 4,NL0 5,NL0 6,NL0 7,NL0 8,NL0 9
Line,,,,,,,,,,
0,0.0,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000e+00,-1.000000,-1.000000,-1.000000
1,0.0,0.493875,0.079274,0.283323,-0.104140,0.022969,6.856901e-17,-0.201976,-0.365029,0.283323
10,0.0,0.219832,0.399912,0.311284,0.045232,0.826128,-3.428450e-16,0.087727,0.158548,0.311284
11,0.0,-0.493875,-0.079274,-0.283323,0.104140,-0.022969,-6.856901e-17,0.201976,-0.634971,-0.283323
2,0.0,0.506125,-0.079274,-0.283323,0.104140,-0.022969,3.428450e-17,0.201976,0.365029,-0.283323
3,0.0,0.219832,0.399912,0.311284,0.045232,-0.173872,-1.028535e-16,0.087727,0.158548,0.311284
4,0.0,0.286293,0.520814,0.405393,0.058907,0.150903,-2.571338e-16,0.114249,0.206481,0.405393
5,0.0,-0.506125,0.079274,-0.716677,-0.104140,0.022969,1.828507e-16,-0.201976,-0.365029,-0.716677
7,0.0,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000


In [13]:
# sanity check: apply a test transfer and see if the resulting flows make sense
# inject +1 at one bus and withdraw -1 at another bus
inj_bus = buses.loc[9, "name"] if len(buses) > 1 else buses.loc[1, "name"]
wdr_bus = ref_bus_name

inj_idx = int(buses.loc[buses["name"] == inj_bus, "bus_idx"].iloc[0])
wdr_idx = int(buses.loc[buses["name"] == wdr_bus, "bus_idx"].iloc[0])

p_test = np.zeros(len(buses))
p_test[inj_idx] = 1.0
p_test[wdr_idx] = -1.0
print("Sum of test injections (should be 0):", p_test.sum())

f_test = PTDF @ p_test

flow_test_df = pd.DataFrame({
    "line": lines["name"],
    "flow_for_test_transfer": f_test
})

print(f"\nTest transfer: +1 MW at {inj_bus}, -1 MW at {wdr_bus}")
display(flow_test_df.head(10))

Sum of test injections (should be 0): 0.0

Test transfer: +1 MW at NL0 9, -1 MW at NL0 0


,line,flow_for_test_transfer
0,0,-1.000000
1,1,0.283323
2,10,0.311284
3,11,-0.283323
4,2,-0.283323
5,3,0.311284
6,4,0.405393
7,5,-0.716677
9,7,0.000000
10,8,0.283323


In [14]:
inj_idx = 8
slack_idx = ref_bus_idx

p = np.zeros(len(buses))
p[inj_idx] = 1.0
p[slack_idx] = -1.0

# remove slack entry from injections
p_red = np.delete(p, slack_idx)

# solve reduced DC load flow for voltage angles
theta_red = np.linalg.solve(B_red, p_red)

# reconstruct full angle vector
theta = np.insert(theta_red, slack_idx, 0.0)

# compute line flows from angles
f_dc = np.diag(b) @ A @ theta

# compute line flows from PTDF
f_ptdf = PTDF @ p

# compare
comparison_df = pd.DataFrame({
    "line": lines["name"].values,
    "flow_ptdf": f_ptdf,
    "flow_dc": f_dc,
    "difference": f_ptdf - f_dc
})

print("Max absolute difference:", np.max(np.abs(f_ptdf - f_dc)))
display(comparison_df)

Max absolute difference: 2.220446049250313e-16


,line,flow_ptdf,flow_dc,difference
0,0,-1.000000,-1.000000,0.000000e+00
1,1,-0.365029,-0.365029,2.220446e-16
2,10,0.158548,0.158548,-5.551115e-17
3,11,-0.634971,-0.634971,0.000000e+00
4,2,0.365029,0.365029,2.220446e-16
5,3,0.158548,0.158548,1.942890e-16
6,4,0.206481,0.206481,2.775558e-17
7,5,-0.365029,-0.365029,-5.551115e-17
8,7,0.000000,0.000000,0.000000e+00
9,8,0.634971,0.634971,1.110223e-16
